# Part 12 (실습) — 왜 LangGraph인가 (고급 시작)

> **이 노트북의 목표**
> 같은 절차 작업("초안→검토→통과 시 발행 / 반려 시 수정 반복")을 `create_agent`로 시켰을 때 흐름이 **보장되지 않는** 것을 관찰하고, 그래프로 그렸을 때 흐름이 **보장되는** 차이를 체감함.
>
> ⚠️ 본격 `StateGraph` 코딩은 Part 13에서 시작함. 이 노트북은 "왜 그래프가 필요한가"를 느끼는 데 집중함.
>
> **사용 모델**: `gemini-3-flash` · **선행**: Part 0~11

## 0. 준비

In [ ]:
# ─── 패키지 설치 ───
# %pip install: 주피터 노트북 안에서 파이썬 패키지를 설치하는 매직 명령어
#   -q (quiet): 설치 로그 최소화 | -U (upgrade): 최신 버전으로 업그레이드
%pip install -q -U langchain langchain-google-genai langgraph

In [ ]:
# os: 파이썬 내장 모듈 — 환경 변수(GOOGLE_API_KEY 등)를 읽고 쓸 때 사용
import os
# getpass: 입력한 글자를 화면에 숨겨서 받는 함수 (비밀번호 입력처럼)
#   → API 키를 노출 없이 안전하게 입력받기 위해 사용
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키: ")
# ChatGoogleGenerativeAI: Google Gemini 모델을 LangChain에서 사용할 수 있게 감싸주는 클래스
#   → langchain-google-genai 패키지에서 제공
#   → .invoke() / .stream() / .batch() 등 LangChain 공통 메서드를 지원
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
# @tool 데코레이터: 일반 파이썬 함수를 LangChain "도구"로 변환
#   → 함수의 docstring이 도구 설명이 되고, 파라미터가 도구 입력 스키마가 됨
#   → 에이전트가 이 설명을 읽고 언제 이 도구를 쓸지 스스로 판단함
from langchain_core.tools import tool
# ─── Gemini 모델 인스턴스 생성 ───
# model: 사용할 Gemini 모델명 (예: "gemini-3-flash" — 빠르고 저렴한 기본 모델)
# temperature: 답변의 무작위성 조절 (0=일관된 답 / 1=창의적·다양한 답)
model = ChatGoogleGenerativeAI(model="gemini-3-flash", temperature=0)
print("준비 완료")

## 1. 우리는 이미 LangGraph 위에 있었다

`create_agent`가 돌려준 객체는 사실 **LangGraph 그래프**임. 구조를 출력해 보면 노드가 보임.

In [ ]:
agent = create_agent(model=model, tools=[], system_prompt="너는 비서야.")

# 에이전트의 내부 그래프 구조 — 노드 목록
print("에이전트 내부 그래프의 노드들:")
print(list(agent.get_graph().nodes.keys()))
print("\n→ create_agent는 미리 만들어진 그래프. 고급에선 이 노드·엣지를 직접 정의함.")

> 📌 `create_agent`는 "agent 노드 ↔ tools 노드를 잇고 조건 반복하는" 기성 그래프임. 메모리(체크포인터)도 LangGraph의 상태 관리였음(Part 10). 우리는 LangGraph를 간접적으로 써 왔음.

## 2. 절차를 에이전트로 시도 — 흐름이 보장되지 않음

"초안→검토→반려 시 수정 반복" 절차를 도구로 주고 에이전트에게 맡겨 봄. 매번 흐름이 달라질 수 있음.

In [ ]:
@tool
def write_draft(topic: str) -> str:
    """주제로 보고서 초안을 작성한다."""
    return f"[초안] {topic}에 대한 보고서 초안입니다."

@tool
def review(draft: str) -> str:
    """초안을 검토해 통과/반려를 판정한다."""
    # 실제론 검토 로직. 여기선 무작위성 흉내
    import random
    return random.choice(["통과", "반려: 근거 부족"])

@tool
def publish(draft: str) -> str:
    """검토를 통과한 문서를 발행한다."""
    return "발행 완료 ✅"

proc_agent = create_agent(
    model=model,
    tools=[write_draft, review, publish],
    system_prompt="초안을 쓰고 검토하고, 통과하면 발행해. 반려되면 수정해서 다시 검토받아.",
)

# 여러 번 실행하면 흐름(도구 호출 순서/횟수)이 매번 다를 수 있음
for run in range(2):
# .invoke(): 입력을 모델/체인에 보내고 결과를 받는 LangChain의 핵심 실행 메서드
#   → 모든 LangChain 부품(Runnable)이 이 메서드를 공유함
    r = proc_agent.invoke({"messages": [("user", "AI 보고서 절차대로 처리해줘")]})
    calls = [tc["name"] for m in r["messages"] if getattr(m,"tool_calls",None) for tc in m.tool_calls]
    print(f"실행 {run+1} 도구 호출 순서: {calls}")
print("\n→ 검토 없이 발행하거나, 반려를 무시하거나, 순서가 매번 다를 수 있음. '반드시'가 보장 안 됨.")

### 무엇이 문제인가
- "검토를 **반드시** 거친다"가 보장 안 됨 — 에이전트가 건너뛸 수 있음.
- "반려되면 **반드시** 수정 루프로" 가 보장 안 됨.
- 매 실행마다 흐름이 달라 **테스트·감사**가 어려움.

→ 이런 "반드시"가 필요한 절차는 흐름을 **명시적으로 그려야** 함.

## 3. 그래프로 그리면 — 흐름이 보장된다 (개념 미리보기)

LangGraph로 같은 절차를 그리면, 흐름이 그림 그대로 보장됨. (실제 코딩은 Part 13)

```
[시작] → (초안 작성) → (검토) → 분기
                                  ├─ 통과 → (발행) → [끝]
                                  └─ 반려 → (수정) → 다시 (검토)로  ← 루프
```

- 검토는 **항상** 거침 (엣지로 고정)
- 반려면 **항상** 수정 루프로 (조건 분기로 고정)
- 발행은 검토 통과 후에만 (엣지로 보장)

매번 같은 흐름이 보장되고, 흐름이 그림으로 명확해 테스트·감사가 쉬움.

In [ ]:
# Part 13에서 직접 구현할 StateGraph의 골격 미리보기 (개념용 의사코드)
sketch = """
from langgraph.graph import StateGraph, START, END

builder = StateGraph(State)
# .add_node(): 그래프에 새로운 노드(작업 단계)를 추가
#   → (노드이름, 실행할함수) 형태로 등록
builder.add_node("draft", draft_node)
builder.add_node("review", review_node)
builder.add_node("revise", revise_node)
builder.add_node("publish", publish_node)

# .add_edge(): 두 노드를 순차적으로 연결 (A → B)
builder.add_edge(START, "draft")
builder.add_edge("draft", "review")
# .add_conditional_edges(): 조건에 따라 다른 노드로 분기하는 엣지 추가
#   → 라우팅 함수의 반환값에 따라 다음 노드가 결정됨
builder.add_conditional_edges("review", route,   # 통과/반려 분기
                              {"pass": "publish", "fail": "revise"})
# .add_edge(): 두 노드를 순차적으로 연결 (A → B)
builder.add_edge("revise", "review")             # 반려→수정→재검토 루프
builder.add_edge("publish", END)

# .compile(): 그래프를 실행 가능한 형태로 컴파일 (체크포인터 연결 등)
graph = builder.compile()
"""
print(sketch)
print("→ 이 골격을 Part 13에서 실제로 짜고 실행함. 흐름이 코드(그림)로 못박힘.")

## 4. 네 신호로 점검하기

이 작업이 왜 그래프인지, 네 신호로 확인.

In [ ]:
signals = {
    "① 반복(루프)": "반려 시 수정→재검토 반복 → ✅ 있음",
    "② 조건 분기": "통과/반려에 따라 다른 길 → ✅ 있음",
    "③ 장기 상태": "초안·검토의견·승인여부를 단계마다 누적 → ✅ 있음",
    "④ 사람 개입": "(이 예시엔 없지만) 발행 전 승인 추가 가능 → Part 14",
}
for k, v in signals.items():
    print(f"{k}: {v}")
print("\n→ ①②③이 강하게 켜짐 → 그래프가 맞음. (④까지 더하면 Part 14 human-in-the-loop)")

## 정리

| 절 | 내용 |
|---|---|
| 1 | create_agent도 사실 LangGraph 그래프 |
| 2 | 절차를 에이전트로 → 흐름 보장 안 됨 |
| 3 | 그래프로 그리면 → 흐름 보장 (Part 13 예고) |
| 4 | 네 신호(반복·분기·상태·개입)로 판단 |

### 3줄 요약
1. create_agent는 기성 그래프 — 흐름은 고정 패턴이라 절차 보장에 한계.
2. "반드시" 거쳐야 하는 절차는 노드·엣지로 직접 그려야 보장됨.
3. 네 신호(반복·분기·상태·개입)가 강하면 그래프로 전환.

### ▶ 다음 (Part 13)
드디어 `StateGraph`를 직접 짬 — 상태 정의 → 노드 → 엣지 → 조건 분기 → 컴파일 → 실행. 네 신호 중 ①②③을 손으로 구현함.